# AYA Edge 0.2 — Tesla T4 architecture and pipeline smoke test

This notebook is a reproducible **smoke-test and prototype-training harness** for the approximately 310M AYA Edge 0.2 candidate. It is not evidence of the approximately 310M target, agent competence, or production readiness. Execute it top-to-bottom on the target Kaggle revision and preserve the printed Git revision and metrics.

## Operating rules

- Detect hardware instead of assuming a Tesla T4 or BF16 support.
- Treat the included 120-example dataset as an overfit/smoke-test dataset only.
- Preserve the Stage 1 experimental direction: 50% general language and 50% instruction/agent patterns.
- Do not report notebook completion as model quality or agent competence.

In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO = WORK / 'Aya-Edge-0.1'
REPO_URL = 'https://github.com/hygienhub03-dev/Aya-Edge-0.1.git'
BRANCH = 'research/tiny-moe-agentic-production'
if not (REPO / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)
print('Working directory:', Path.cwd())
print('Git branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
assert subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip() == BRANCH
print('Git revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## 1. Environment inspection

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
    print('FP16 supported: yes')
    print('BF16 supported:', torch.cuda.is_bf16_supported())
else:
    print('GPU unavailable: running CPU smoke test')
print('System:', platform.platform())
print('Disk free GiB:', round(shutil.disk_usage(WORK).free / 2**30, 2))

## 2. Repository setup and tests

In [ ]:
test_result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], text=True, capture_output=True)
print(test_result.stdout)
print(test_result.stderr)
if test_result.returncode != 0:
    raise RuntimeError('Repository tests failed; stop before training.')

## 3. Dataset inspection

In [ ]:
from collections import Counter
DATA = REPO / 'aya_edge_mini_dataset_v1'
manifest = json.loads((DATA / 'dataset_manifest.json').read_text())
print(json.dumps(manifest, indent=2))
for split in ('train', 'validation', 'test'):
    rows = [json.loads(line) for line in (DATA / f'{split}.jsonl').read_text().splitlines()]
    print(split, 'records=', len(rows), 'categories=', Counter(r['category'] for r in rows))
print('Dataset note: simulated tool results and 120 records make this suitable only for smoke/overfit checks.')

## 4. Model construction and parameter report

In [ ]:
import yaml
from aya_edge import AyaEdgeConfig, AyaEdgeForCausalLM, count_active_parameters, count_parameters, set_seed
raw = yaml.safe_load((REPO / 'configs/aya_edge_0_2.yaml').read_text())
cfg = AyaEdgeConfig(**raw['model'])
set_seed(raw['training']['seed'])
model = AyaEdgeForCausalLM(cfg).to(device)
print('Configuration:', cfg.to_dict())
print('Total trainable parameters:', f'{count_parameters(model):,}')
print('Structural active-parameter estimate:', f'{count_active_parameters(model):,}')
print('Prototype status: not production training; active estimate is not a FLOP count.')

## 5. Deterministic byte-level smoke adapter

The repository does not include a tokenizer package. This adapter only exercises model and training plumbing; it is not the final tokenizer or a language-quality experiment.

In [ ]:
def load_split(name, seq_len=128):
    rows = [json.loads(line) for line in (DATA / f'{name}.jsonl').read_text().splitlines()]
    stream = []
    for row in rows:
        stream.extend(b + 1 for b in row['text'].encode('utf-8'))
    usable = (len(stream) // seq_len) * seq_len
    if usable == 0:
        raise ValueError(f'{name} split has fewer than one packed sequence')
    # Fixed-length packing drops only the incomplete tail; no artificial pad IDs enter loss or attention.
    return torch.tensor(stream[:usable], dtype=torch.long).view(-1, seq_len)
train_tokens = load_split('train', min(cfg.max_seq_len, 128))
val_tokens = load_split('validation', min(cfg.max_seq_len, 128))
print('Packed train:', tuple(train_tokens.shape), 'Packed validation:', tuple(val_tokens.shape))

## 6. Conservative training configuration

In [ ]:
from torch.amp import GradScaler, autocast
from aya_edge import save_checkpoint
train_cfg = raw['training']
optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg['learning_rate'], weight_decay=train_cfg['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, min(train_cfg['max_steps'], 100)))
use_amp = device.type == 'cuda'
scaler = GradScaler('cuda', enabled=use_amp)
batch_size = min(int(train_cfg['batch_size']), 2)
max_steps = min(int(train_cfg['max_steps']), 20)
print({'batch_size': batch_size, 'max_steps': max_steps, 'amp': use_amp})

## 7. Forward/backward smoke test

In [ ]:
batch = train_tokens[:batch_size].to(device)
optimizer.zero_grad(set_to_none=True)
with autocast('cuda', dtype=torch.float16, enabled=use_amp):
    out = model(batch, labels=batch)
    total_loss = out['loss'] + train_cfg['aux_loss_weight'] * out['aux_loss']
scaler.scale(total_loss).backward()
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
finite = all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
scaler.step(optimizer); scaler.update(); scheduler.step()
print('logits:', tuple(out['logits'].shape), 'loss:', float(out['loss']), 'aux:', float(out['aux_loss']))
print('gradient norm:', float(grad_norm), 'finite gradients:', finite)

## 8. Short training loop, validation, routing, and checkpoint

In [ ]:
def evaluate(tokens):
    model.eval()
    with torch.no_grad():
        value = model(tokens.to(device), labels=tokens.to(device))
    model.train()
    return float(value['loss']), float(value['aux_loss'])

start = time.time()
for step in range(1, max_steps + 1):
    index = ((step - 1) * batch_size) % len(train_tokens)
    batch = train_tokens[index:index + batch_size].to(device)
    optimizer.zero_grad(set_to_none=True)
    with autocast('cuda', dtype=torch.float16, enabled=use_amp):
        out = model(batch, labels=batch)
        loss = out['loss'] + train_cfg['aux_loss_weight'] * out['aux_loss']
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    if not torch.isfinite(grad_norm):
        raise FloatingPointError('non-finite gradient norm')
    scaler.step(optimizer); scaler.update(); scheduler.step()
    if step == 1 or step % 5 == 0:
        val_loss, val_aux = evaluate(val_tokens[:batch_size])
        elapsed = max(time.time() - start, 1e-9)
        print({'step': step, 'train_loss': float(out['loss']), 'val_loss': val_loss,
               'aux_loss': float(out['aux_loss']), 'val_aux': val_aux,
               'tokens_per_sec': round(step * batch.numel() / elapsed, 1),
               'peak_memory_gib': round(torch.cuda.max_memory_allocated()/2**30, 3) if device.type == 'cuda' else None})

save_checkpoint(WORK / 'aya_edge_t4_latest.pt', model, optimizer, scheduler, scaler,
                step=max_steps, tokens_processed=max_steps * batch.numel(), config=cfg,
                dataset_version='aya_edge_mini_dataset_v1')
print('Checkpoint:', WORK / 'aya_edge_t4_latest.pt')

## 9. Final report and limitations

In [ ]:
print('Final revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Model parameters:', count_parameters(model))
print('Dataset:', manifest.get('version', 'unknown'))
print('Output directory:', WORK)
for item in [
    'The byte adapter is a deterministic development adapter, not the approved 32,768-vocabulary production tokenizer.',
    'The mini dataset is a development fixture and is not a representative production corpus.',
    'The candidate still uses a readable Python-loop MoE dispatcher and MLA-inspired attention; production kernels and KV-cache measurements remain future work.',
    'A successful run does not establish agent competence, reasoning quality, or production readiness.',
]:
    print('-', item)